# Module 3 - Class 2: Scaling and Encoding Experiment

**Objective:** See firsthand how feature scaling and encoding choices affect model performance.

### What you will practice
- Training KNN with and without scaling
- Comparing MinMaxScaler vs StandardScaler
- Label encoding vs One-Hot encoding

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 1. Load and Prepare Data

We load the Telco Churn dataset and do minimal cleaning (same as Class 1).

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Standardize verbose 'No X service' values
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Drop customerID
df = df.drop('customerID', axis=1)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Separate numeric features for the scaling experiment
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_numeric = df[numeric_cols]
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X_numeric, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. KNN Without Scaling

In [ ]:
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
y_pred_raw = knn_raw.predict(X_test)

acc_raw = accuracy_score(y_test, y_pred_raw)
print(f"KNN accuracy (no scaling): {acc_raw:.4f}")
print()
print(classification_report(y_test, y_pred_raw))

Notice how raw features have very different scales:
- `tenure`: 0-72
- `MonthlyCharges`: ~18-118
- `TotalCharges`: 0-8600+

KNN uses distance, so `TotalCharges` dominates the distance calculation.

## 3. KNN with MinMaxScaler

In [ ]:
scaler_mm = MinMaxScaler()
X_train_mm = scaler_mm.fit_transform(X_train)
X_test_mm = scaler_mm.transform(X_test)

knn_mm = KNeighborsClassifier(n_neighbors=5)
knn_mm.fit(X_train_mm, y_train)
y_pred_mm = knn_mm.predict(X_test_mm)

acc_mm = accuracy_score(y_test, y_pred_mm)
print(f"KNN accuracy (MinMaxScaler): {acc_mm:.4f}")
print()
print(classification_report(y_test, y_pred_mm))

## 4. KNN with StandardScaler

In [ ]:
scaler_ss = StandardScaler()
X_train_ss = scaler_ss.fit_transform(X_train)
X_test_ss = scaler_ss.transform(X_test)

knn_ss = KNeighborsClassifier(n_neighbors=5)
knn_ss.fit(X_train_ss, y_train)
y_pred_ss = knn_ss.predict(X_test_ss)

acc_ss = accuracy_score(y_test, y_pred_ss)
print(f"KNN accuracy (StandardScaler): {acc_ss:.4f}")
print()
print(classification_report(y_test, y_pred_ss))

## 5. Scaling Comparison Table

In [ ]:
results = pd.DataFrame({
    'Method': ['No Scaling', 'MinMaxScaler', 'StandardScaler'],
    'Accuracy': [acc_raw, acc_mm, acc_ss]
})
results['Accuracy'] = results['Accuracy'].round(4)
results

**Key takeaway:** Distance-based algorithms like KNN are sensitive to feature scales. Scaling brings all features to the same range, preventing any single feature from dominating.

---

## 6. TODO: Label Encoding vs One-Hot Encoding Comparison

Now let's see how encoding categorical features affects the model.

**Your tasks:**
1. Prepare the full dataset (numeric + categorical features) using Label Encoding
2. Prepare the full dataset using One-Hot Encoding
3. Train KNN on both (with StandardScaler) and compare accuracy

In [ ]:
# Identify categorical columns (excluding target)
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

In [ ]:
# TODO Part A: Label Encoding
# 1. Create a copy of df
# 2. Use LabelEncoder to encode each categorical column
# 3. Split into train/test (test_size=0.2, random_state=42, stratify=y)
# 4. Scale with StandardScaler
# 5. Train KNN(n_neighbors=5) and compute accuracy

df_label = df.copy()
# YOUR CODE HERE


acc_label = None  # Replace with your accuracy
print(f"KNN accuracy (Label Encoding + StandardScaler): {acc_label}")

In [ ]:
# TODO Part B: One-Hot Encoding
# 1. Use pd.get_dummies(df, columns=cat_cols, drop_first=True)
# 2. Split into train/test (same params)
# 3. Scale with StandardScaler
# 4. Train KNN(n_neighbors=5) and compute accuracy

df_onehot = pd.get_dummies(df, columns=cat_cols, drop_first=True)
# YOUR CODE HERE


acc_onehot = None  # Replace with your accuracy
print(f"KNN accuracy (One-Hot Encoding + StandardScaler): {acc_onehot}")

In [ ]:
# TODO Part C: Build the final comparison table
# Include all 5 experiments: 
#   No Scaling, MinMaxScaler, StandardScaler, Label Encoding, One-Hot Encoding

final_results = pd.DataFrame({
    'Experiment': [
        'Numeric Only - No Scaling',
        'Numeric Only - MinMaxScaler',
        'Numeric Only - StandardScaler',
        'All Features - Label Encoding + StandardScaler',
        'All Features - One-Hot Encoding + StandardScaler'
    ],
    'Accuracy': [acc_raw, acc_mm, acc_ss, acc_label, acc_onehot]
})
final_results

---
## Summary

| Concept | Key Point |
|---------|-----------|
| No scaling | Distance-based models suffer from scale differences |
| MinMaxScaler | Scales to [0, 1] range; sensitive to outliers |
| StandardScaler | Scales to mean=0, std=1; more robust to outliers |
| Label Encoding | Maps categories to integers; implies ordinal relationship |
| One-Hot Encoding | Creates binary columns; no implied ordering |